# Proyecto Final Integrador | Data Analytics

**Estudiante:** Bezerra, Roberto  
**Comisión:** TT26109 | TPI Data Analytics

## Objetivo
Analizar datos de clientes, ventas y marketing para identificar tendencias, productos de alto rendimiento y oportunidades.

## Etapa 1 | Recopilación y preparación

### 1. Carga de los DataFrames
El notebook está preparado para ejecutarse en Google Colab con fuentes públicas de Google Drive.

In [ ]:
# Importar las bibliotecas necesarias.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)

# Definir una función reutilizable para cargar CSV públicos de Google Drive.
def cargar_dataset(enlace):
    file_id = enlace.split("/d/")[1].split("/")[0]
    return pd.read_csv(f"https://drive.google.com/uc?id={file_id}")

fuentes = {
    "clientes": "https://drive.google.com/file/d/1pyNXpqnOauR5tWJWj9f8C15Ygn5HmdWT/view?usp=drive_link",
    "ventas": "https://drive.google.com/file/d/1dpvV5iOAKyD7QZX8gwAZVZIefWGpnTKM/view?usp=drive_link",
    "marketing": "https://drive.google.com/file/d/1NV-ICitmc8HQjhNEuUcgKqvajoBbNBT5/view?usp=drive_link",
}
df_clientes = cargar_dataset(fuentes["clientes"])
df_ventas = cargar_dataset(fuentes["ventas"])
df_marketing = cargar_dataset(fuentes["marketing"])

# Verificar dimensiones y ejemplos de los datos cargados.
for nombre, dataframe in {"Clientes": df_clientes, "Ventas": df_ventas, "Marketing": df_marketing}.items():
    print(f"\n{nombre}: {dataframe.shape[0]} filas y {dataframe.shape[1]} columnas")
    display(dataframe.head(3))

### 2. Script básico y estructura de datos
Se emplean variables y operadores para calcular ventas; se utiliza una **lista de diccionarios** porque cada venta conserva campos con nombre.

In [ ]:
# Calcular las ventas mensuales de diez productos con variables y operadores.
ventas_ejemplo = [
    {"producto":"Lámpara de mesa", "precio":105.10, "cantidad":5}, {"producto":"Cuadro decorativo", "precio":69.94, "cantidad":5},
    {"producto":"Tablet", "precio":320.00, "cantidad":2}, {"producto":"Auriculares", "precio":85.50, "cantidad":8},
    {"producto":"Silla de oficina", "precio":190.00, "cantidad":3}, {"producto":"Monitor", "precio":240.00, "cantidad":4},
    {"producto":"Mouse inalámbrico", "precio":42.00, "cantidad":12},
    {"producto":"Mochila", "precio":68.00, "cantidad":6}, {"producto":"Botella térmica", "precio":31.50, "cantidad":15},
]
df_ventas_ejemplo = pd.DataFrame(ventas_ejemplo)
df_ventas_ejemplo["venta_mensual"] = df_ventas_ejemplo["precio"] * df_ventas_ejemplo["cantidad"]
venta_mensual_total = df_ventas_ejemplo["venta_mensual"].sum()
print(f"Ventas mensuales totales de los 10 productos: ${venta_mensual_total:,.2f}")
display(df_ventas_ejemplo.sort_values("venta_mensual", ascending=False))

### 3. EDA inicial y calidad de datos
Se revisan dimensiones, tipos, nulos y duplicados antes de modificar los datos.


In [ ]:
# Crear un perfil visual y compacto para el diagnóstico inicial de calidad.
def perfil_inicial(nombre, dataframe):
    resumen = pd.DataFrame({"Métrica":["Filas", "Columnas", "Duplicados exactos", "Celdas nulas"], "Valor":[len(dataframe), dataframe.shape[1], int(dataframe.duplicated().sum()), int(dataframe.isna().sum().sum())]})
    perfil = pd.DataFrame({"tipo_de_dato":dataframe.dtypes.astype(str), "nulos":dataframe.isna().sum(), "nulos_pct":(dataframe.isna().mean()*100).round(2), "valores_unicos":dataframe.nunique()}).sort_values("nulos_pct", ascending=False)
    print(f"\nEDA INICIAL | {nombre.upper()}")
    display(resumen)
    display(perfil.style.format({"nulos_pct":"{:.2f}%"}).background_gradient(subset=["nulos_pct"], cmap="YlOrRd"))
    display(dataframe.head(3).style.set_caption(f"Muestra de {nombre}"))
    return perfil
perfiles_iniciales = {nombre: perfil_inicial(nombre, dataframe) for nombre, dataframe in {"clientes":df_clientes, "ventas":df_ventas, "marketing":df_marketing}.items()}

### Visualizaciones del diagnóstico inicial
Estos gráficos complementan la evaluación de calidad y distribuciones antes de la limpieza.

In [ ]:
# Visualizar calidad y distribuciones iniciales.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(data=df_clientes, x="ingresos", bins=25, kde=True, color="#2563eb", ax=axes[0]); axes[0].set(title="Distribución de ingresos de clientes")
nulos = df_ventas.isna().sum(); nulos = nulos[nulos > 0]
sns.barplot(x=nulos.index, y=nulos.values, color="#f97316", ax=axes[1]); axes[1].set(title="Valores nulos en ventas", xlabel="Campo", ylabel="Registros")
axes[1].tick_params(axis="x", rotation=30)
conteo = df_ventas["categoria"].value_counts()
sns.barplot(x=conteo.index, y=conteo.values, color="#16a34a", ax=axes[2]); axes[2].set(title="Operaciones por categoría", xlabel="Categoría", ylabel="Operaciones")
plt.tight_layout(); plt.show()

**Estado inicial:** clientes y marketing no presentan nulos ni duplicados exactos. Ventas contiene nulos en `precio` y `cantidad`, duplicados exactos y tipos que requieren normalización.

## Etapa 2 | Preprocesamiento, agregación e integración

In [ ]:
# Crear copias, normalizar encabezados y quitar espacios de textos.
df_clientes_limpio, df_ventas_limpio, df_marketing_limpio = df_clientes.copy(), df_ventas.copy(), df_marketing.copy()
for dataframe in [df_clientes_limpio, df_ventas_limpio, df_marketing_limpio]:
    dataframe.columns = dataframe.columns.str.strip().str.lower()
    for columna in dataframe.select_dtypes(include="object").columns:
        dataframe[columna] = dataframe[columna].str.strip()

# Normalizar ventas: precio numérico, fecha válida, duplicados y nulos.
filas_ventas_iniciales = len(df_ventas_limpio)
duplicados_ventas = df_ventas_limpio.duplicated().sum()
df_ventas_limpio["precio"] = df_ventas_limpio["precio"].astype("string").str.replace(r"[^0-9,.-]", "", regex=True).str.replace(",", ".", regex=False)
df_ventas_limpio["precio"] = pd.to_numeric(df_ventas_limpio["precio"], errors="coerce")
df_ventas_limpio["cantidad"] = pd.to_numeric(df_ventas_limpio["cantidad"], errors="coerce")
df_ventas_limpio["fecha_venta"] = pd.to_datetime(df_ventas_limpio["fecha_venta"], format="%d/%m/%Y", errors="coerce")
df_ventas_limpio = df_ventas_limpio.drop_duplicates().dropna(subset=["producto", "categoria", "precio", "cantidad", "fecha_venta"]).copy()
df_ventas_limpio["cantidad"] = df_ventas_limpio["cantidad"].astype(int)
df_ventas_limpio["ingreso"] = df_ventas_limpio["precio"]
df_ventas_limpio["mes"] = df_ventas_limpio["fecha_venta"].dt.to_period("M").astype(str)

# Normalizar marketing y calcular duración de campañas.
for columna in ["fecha_inicio", "fecha_fin"]:
    df_marketing_limpio[columna] = pd.to_datetime(df_marketing_limpio[columna], format="%d/%m/%Y", errors="coerce")
df_marketing_limpio["costo"] = pd.to_numeric(df_marketing_limpio["costo"], errors="coerce")
df_marketing_limpio = df_marketing_limpio.drop_duplicates().dropna(subset=["producto", "canal", "costo", "fecha_inicio", "fecha_fin"]).copy()
df_marketing_limpio["duracion_dias"] = (df_marketing_limpio["fecha_fin"] - df_marketing_limpio["fecha_inicio"]).dt.days
df_clientes_limpio = df_clientes_limpio.drop_duplicates().copy()

# Validar el resultado de la limpieza.
validacion_limpieza = pd.DataFrame([{"dataset": n, "filas_finales": len(d), "nulos_totales": int(d.isna().sum().sum()), "duplicados_exactos": int(d.duplicated().sum())} for n, d in {"clientes": df_clientes_limpio, "ventas": df_ventas_limpio, "marketing": df_marketing_limpio}.items()])
print(f"Duplicados eliminados en ventas: {duplicados_ventas}")
print(f"Filas removidas en ventas: {filas_ventas_iniciales - len(df_ventas_limpio)}")
display(validacion_limpieza)
assert validacion_limpieza["nulos_totales"].eq(0).all()
assert validacion_limpieza["duplicados_exactos"].eq(0).all()

In [ ]:
# Agregar ventas por producto y definir alto rendimiento como el cuartil superior.
ventas_por_producto = df_ventas_limpio.groupby(["producto", "categoria"], as_index=False).agg(ingresos=("ingreso", "sum"), unidades=("cantidad", "sum"), operaciones=("id_venta", "nunique"))
umbral_alto_rendimiento = ventas_por_producto["ingresos"].quantile(0.75)
productos_alto_rendimiento = ventas_por_producto.query("ingresos >= @umbral_alto_rendimiento").sort_values("ingresos", ascending=False)
ventas_por_categoria = df_ventas_limpio.groupby("categoria", as_index=False).agg(ingresos=("ingreso", "sum"), unidades=("cantidad", "sum"), operaciones=("id_venta", "nunique")).sort_values("ingresos", ascending=False)
ventas_por_categoria["participacion_ingresos_pct"] = (100 * ventas_por_categoria["ingresos"] / ventas_por_categoria["ingresos"].sum()).round(2)
print(f"Umbral de alto rendimiento: ${umbral_alto_rendimiento:,.2f}")
display(productos_alto_rendimiento)
display(ventas_por_categoria)

# Agregar marketing por producto antes de integrar para no multiplicar ventas.
marketing_por_producto = df_marketing_limpio.groupby("producto", as_index=False).agg(costo_marketing=("costo", "sum"), campanas=("id_campanha", "nunique"), canales=("canal", "nunique"))
analisis_producto = ventas_por_producto.merge(marketing_por_producto, on="producto", how="left", validate="one_to_one")
analisis_producto[["costo_marketing", "campanas", "canales"]] = analisis_producto[["costo_marketing", "campanas", "canales"]].fillna(0)
analisis_producto["ratio_ingreso_costo"] = np.where(analisis_producto["costo_marketing"] > 0, analisis_producto["ingresos"] / analisis_producto["costo_marketing"], np.nan)
analisis_producto = analisis_producto.sort_values("ingresos", ascending=False)
display(analisis_producto.head(10))

## Etapa 3 | Análisis de datos

### Estadística descriptiva, EDA y correlación

In [ ]:
# Calcular estadística descriptiva, evolución mensual y correlación.
estadistica_ventas = df_ventas_limpio[["precio", "cantidad", "ingreso"]].agg(["count", "mean", "median", "std", "min", "max"]).T.round(2)
ventas_mensuales = df_ventas_limpio.groupby("mes", as_index=False).agg(ingresos=("ingreso", "sum"), unidades=("cantidad", "sum"), operaciones=("id_venta", "nunique")).sort_values("mes")
correlacion_precio_cantidad = df_ventas_limpio[["precio", "cantidad"]].corr().loc["precio", "cantidad"]
display(estadistica_ventas)
print(f"Correlación precio-cantidad: {correlacion_precio_cantidad:.3f}")

# Crear gráficos de línea, barras y dispersión.
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
axes[0].plot(ventas_mensuales["mes"], ventas_mensuales["ingresos"], marker="o", color="#1f77b4")
axes[0].set(title="Ingresos mensuales", xlabel="Mes", ylabel="Ingresos ($)")
axes[0].tick_params(axis="x", rotation=45)
sns.barplot(data=ventas_por_categoria, x="categoria", y="ingresos", ax=axes[1], color="#f58518")
axes[1].set(title="Ingresos por categoría", xlabel="Categoría", ylabel="Ingresos ($)")
sns.regplot(data=df_ventas_limpio, x="precio", y="cantidad", ax=axes[2], scatter_kws={"alpha": 0.35, "color": "#4c78a8"}, line_kws={"color": "#f58518"})
axes[2].set(title="Precio y cantidad vendida", xlabel="Precio ($)", ylabel="Cantidad")
plt.tight_layout()
plt.show()

### Visualizaciones complementarias
Se incluyen matriz de correlación, boxplot y ranking de productos para enriquecer la presentación.


In [ ]:
# Incorporar gráficos estadísticos adicionales.
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.heatmap(df_ventas_limpio[["precio", "cantidad", "ingreso"]].corr(), annot=True, fmt=".2f", cmap="Blues", vmin=-1, vmax=1, ax=axes[0]); axes[0].set_title("Matriz de correlación")
sns.boxplot(data=df_ventas_limpio, x="categoria", y="precio", color="#93c5fd", ax=axes[1]); axes[1].set(title="Distribución de precio por categoría", xlabel="Categoría", ylabel="Precio ($)"); axes[1].tick_params(axis="x", rotation=25)
top = analisis_producto.head(10).sort_values("ingresos")
sns.barplot(data=top, y="producto", x="ingresos", color="#7c3aed", ax=axes[2]); axes[2].set(title="Top 10 productos por ingresos", xlabel="Ingresos ($)")
plt.tight_layout(); plt.show()

## Etapa 4 | Visualización, dashboard y conjunto final

In [ ]:
# Consolidar el dataset final y lo exportar para la presentación.
dataset_final = analisis_producto.merge(productos_alto_rendimiento[["producto"]].assign(alto_rendimiento=True), on="producto", how="left")
dataset_final["alto_rendimiento"] = dataset_final["alto_rendimiento"].fillna(False)
dataset_final = dataset_final.sort_values("ingresos", ascending=False).reset_index(drop=True)
dataset_final.to_csv("dataset_final_consolidado.csv", index=False)
display(dataset_final.head(10))

# Construir un dashboard interactivo con Plotly.
dashboard = make_subplots(rows=2, cols=2, subplot_titles=("Ingresos mensuales", "Ingresos por categoría", "Precio y cantidad", "Ingresos por producto"), specs=[[{"type": "scatter"}, {"type": "bar"}], [{"type": "scatter"}, {"type": "bar"}]])
dashboard.add_trace(go.Scatter(x=ventas_mensuales["mes"], y=ventas_mensuales["ingresos"], mode="lines+markers"), row=1, col=1)
dashboard.add_trace(go.Bar(x=ventas_por_categoria["categoria"], y=ventas_por_categoria["ingresos"]), row=1, col=2)
dashboard.add_trace(go.Scatter(x=df_ventas_limpio["precio"], y=df_ventas_limpio["cantidad"], mode="markers", text=df_ventas_limpio["producto"], marker={"color": df_ventas_limpio["categoria"].astype("category").cat.codes, "colorscale": "Viridis", "showscale": False}), row=2, col=1)
dashboard.add_trace(go.Bar(x=analisis_producto["producto"], y=analisis_producto["ingresos"]), row=2, col=2)
dashboard.update_layout(height=850, width=1150, title_text="Dashboard comercial: ventas y marketing", showlegend=False)
dashboard.update_xaxes(tickangle=45, row=1, col=1)
dashboard.update_xaxes(tickangle=45, row=2, col=2)
dashboard.show()

## Conclusiones

In [ ]:
# Generar indicadores reproducibles para la presentación final.
producto_lider = dataset_final.iloc[0]
categoria_lider = ventas_por_categoria.iloc[0]
mes_lider = ventas_mensuales.loc[ventas_mensuales["ingresos"].idxmax()]
print(f"• Producto con mayores ingresos: {producto_lider['producto']} (${producto_lider['ingresos']:,.2f}).")
print(f"• Categoría líder: {categoria_lider['categoria']} ({categoria_lider['participacion_ingresos_pct']:.2f}% de los ingresos).")
print(f"• Mes con mayores ingresos: {mes_lider['mes']} (${mes_lider['ingresos']:,.2f}).")
print(f"• Correlación precio-cantidad: {correlacion_precio_cantidad:.3f}.")
print("• La correlación describe asociación, no causalidad.")

## Anexo

1. `clientes.csv`, `ventas.csv` y `marketing.csv`: fuentes públicas enlazadas en la carga.
2. `dataset_final_consolidado.csv`: archivo generado por el notebook.